# Final IBM Quantum Level-2 Sweet-Spot Evaluation

## Objective

This notebook performs the **final research evaluation** of the Temporal 30-day Sweet-Spot experiment against topology-valid random 3-qubit controls on:

- `ibm_fez`
- `ibm_kingston`
- `ibm_marrakesh`

It is designed to:

1. collect/recover the completed IBM Runtime jobs without resubmitting circuits;
2. combine the FEZ, Kingston, and Marrakesh hardware results;
3. evaluate GHZ fidelity (`P(000) + P(111)`);
4. compare the model-selected Sweet-Spot chain against random controls **within each backend**;
5. quantify effect size, improvement, rank, percentile, bootstrap confidence intervals, and paired differences;
6. test whether the observed advantage survives a depth-matched sensitivity analysis;
7. compare the **1-day / 3-day / 7-day / 30-day temporal model-selection experiments**;
8. explain why the 30-day model was selected as the temporal sweet spot and why the 7-day model was not selected;
9. test whether the final hardware evidence supports the claim that the selected Sweet-Spot chain is better than random controls;
10. generate publication/report-ready CSV/JSON tables and figures.

> **Important scientific distinction:** the model predicts *hardware viability from calibration telemetry*. Hardware GHZ fidelity is the downstream experimental outcome. A high viability score is not itself a fidelity measurement. The experiment tests whether selecting qubits using the model produces better hardware outcomes than topology-valid random selection.

The notebook does **not** submit new QPU jobs.


## Experimental definition

### Sweet Spot

For the final experiment, the **Temporal 30-day model** is the selected model. For each backend, it predicts a viability score for every qubit using the previous 30 observed days of telemetry. All topology-valid physical 3-qubit chains are then enumerated and scored by the mean predicted viability of their three qubits. The highest-scoring valid chain is the Sweet-Spot chain.

The exact binary training label is:

- `T1_us > 100`
- `T2_us > 50`
- `readout_error < 0.05`

A qubit is viable only when all three conditions hold.

The final hardware experiment then compares:

**1 Sweet-Spot chain + 10 topology-valid random controls per backend**, with 256 shots per circuit.

### Primary hardware endpoint

For a GHZ circuit, the primary fidelity used here is:

`F_GHZ = [N(000) + N(111)] / shots`

This is intentionally evaluated directly from the observed counts.


In [7]:
# Core imports
from pathlib import Path
import json, os, re, math, warnings, time
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.special import betaincinv

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Project-relative locations are preferred.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "results").exists():
    # Common local-development fallback used by the supplied files.
    candidates = [
        Path(r"D:/Downloads/uday projects/ibm_circuits"),
        Path("/mnt/data"),
    ]
    for p in candidates:
        if p.exists() and (p / "results").exists():
            PROJECT_ROOT = p
            break

RESULTS_DIR = PROJECT_ROOT / "results"
OUT_DIR = RESULTS_DIR / "sweet_spot_evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)
print("OUT_DIR     :", OUT_DIR)


PROJECT_ROOT: D:\Downloads\uday projects\ibm_circuits
RESULTS_DIR : D:\Downloads\uday projects\ibm_circuits\results
OUT_DIR     : D:\Downloads\uday projects\ibm_circuits\results\sweet_spot_evaluation


## 1. Final experiment constants and known IBM job IDs

The two Kingston jobs are treated separately:

- the **new Kingston job** is the primary final-experiment result;
- the earlier Kingston job is retained as a **repeatability / replication observation**.

Do not average the two Kingston runs and call the result an independent backend replicate. They are repeated measurements on the same backend.

If the result JSONs already exist locally, the notebook uses them first. If they do not, the IBM collection cell can retrieve completed jobs from IBM Quantum using the saved job IDs or automatic Kingston discovery.


In [8]:
# Known completed jobs from the final experiment.
# FEZ and Marrakesh IDs are fixed from the completed final run.
# Kingston contains TWO completed jobs; the notebook can discover the two
# newest completed Kingston jobs after the experiment start if the IDs
# are not available in a local submission/result checkpoint.

KNOWN_JOB_IDS = {
    "ibm_fez": ["dafgfb5nj4cs73agbfo0"],
    "ibm_marrakesh": ["dafpe83dd5gc73da62o0"],
    "ibm_kingston": [],  # auto-discover the two newest completed Kingston jobs
}

# Optional manual override if you have the full Kingston IDs.
# Example:
# KNOWN_JOB_IDS["ibm_kingston"] = ["FULL_NEW_KINGSTON_ID", "FULL_OLD_KINGSTON_ID"]

BACKENDS = ["ibm_fez", "ibm_kingston", "ibm_marrakesh"]
SHOTS_EXPECTED = 256
EXPECTED_CIRCUITS_PER_JOB = 11
FINAL_MODEL = "temporal_30day"
PREDICTION_DATE = "2026-09-06"

print("Final model:", FINAL_MODEL)
print("Prediction date:", PREDICTION_DATE)


Final model: temporal_30day
Prediction date: 2026-09-06


## 2. Locate existing result files

The final runner was designed to save per-backend result JSONs immediately after completion. This cell searches the project for those files.

If they exist, **no IBM API call is needed to recover the measurements**. This is safer and reproducible because the raw counts are already persisted locally.


In [9]:
def find_first(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

RESULT_FILES = {
    "ibm_fez": find_first(
        RESULTS_DIR / "sweet_spot_level2_ibm_fez_result.json",
        PROJECT_ROOT / "sweet_spot_level2_ibm_fez_result.json",
        Path("/mnt/data/sweet_spot_level2_ibm_fez_result.json"),
    ),
    "ibm_kingston": find_first(
        RESULTS_DIR / "sweet_spot_level2_ibm_kingston_result.json",
        PROJECT_ROOT / "sweet_spot_level2_ibm_kingston_result.json",
        Path("/mnt/data/sweet_spot_level2_ibm_kingston_result.json"),
    ),
    "ibm_marrakesh": find_first(
        RESULTS_DIR / "sweet_spot_level2_ibm_marrakesh_result.json",
        PROJECT_ROOT / "sweet_spot_level2_ibm_marrakesh_result.json",
        Path("/mnt/data/sweet_spot_level2_ibm_marrakesh_result.json"),
    ),
}

for b, p in RESULT_FILES.items():
    print(f"{b:16s} -> {p if p else 'NOT FOUND'}")


ibm_fez          -> D:\Downloads\uday projects\ibm_circuits\results\sweet_spot_level2_ibm_fez_result.json
ibm_kingston     -> D:\Downloads\uday projects\ibm_circuits\results\sweet_spot_level2_ibm_kingston_result.json
ibm_marrakesh    -> D:\Downloads\uday projects\ibm_circuits\results\sweet_spot_level2_ibm_marrakesh_result.json


## 3. Optional IBM recovery — no resubmission

If any final result JSON is missing, this cell can retrieve the already-completed Runtime job from IBM Quantum.

IBM documents `service.job(job_id)` for retrieving a known job and `service.jobs(backend_name=...)` for discovering jobs on a backend. A completed `RuntimeJobV2` is in the `DONE` final state and its `result()` returns the primitive results. citeturn0search0turn1search1

**This cell never calls `SamplerV2.run()` and therefore never submits a new QPU job.**


In [10]:
def extract_counts(pub_result):
    data = pub_result.data

    if hasattr(data, "c"):
        counts = data.c.get_counts()
        return {str(k): int(v) for k, v in counts.items()}

    for name in dir(data):
        if name.startswith("_"):
            continue
        try:
            obj = getattr(data, name)
            if hasattr(obj, "get_counts"):
                counts = obj.get_counts()
                return {str(k): int(v) for k, v in counts.items()}
        except Exception:
            pass

    raise RuntimeError("Could not extract SamplerV2 measurement counts.")

def ghz_fidelity(counts):
    total = int(sum(counts.values()))
    if total <= 0:
        raise ValueError("No measurement shots.")
    return (int(counts.get("000", 0)) + int(counts.get("111", 0))) / total

def job_result_to_counts(result):
    if len(result) != EXPECTED_CIRCUITS_PER_JOB:
        raise ValueError(f"Expected {EXPECTED_CIRCUITS_PER_JOB} PUB results, got {len(result)}")
    return [extract_counts(result[i]) for i in range(len(result))]

def discover_recent_kingston_jobs(service):
    jobs = service.jobs(backend_name="ibm_kingston", limit=20, descending=True)
    rows = []
    for job in jobs:
        try:
            status = str(job.status())
        except Exception:
            status = "UNKNOWN"
        try:
            created = job.creation_date
        except Exception:
            created = None
        rows.append({
            "job_id": job.job_id(),
            "status": status,
            "created": created,
        })
    df = pd.DataFrame(rows)
    display(df)

    # Prefer completed jobs created during/after the final experiment start.
    # The screenshot shows the two relevant Kingston jobs at 00:21 and 10:54
    # on 2026-09-08 local display time.
    if not df.empty:
        done = df[df["status"].str.upper().str.contains("DONE|COMPLETED", regex=True)].copy()
        if not done.empty:
            return done.head(2)["job_id"].tolist()
    return []

def retrieve_job_to_raw_json(service, backend, job_id):
    job = service.job(job_id)
    status = str(job.status())
    print(f"{backend}: {job_id} -> {status}")

    if status.upper() not in {"DONE", "COMPLETED"}:
        raise RuntimeError(f"Job {job_id} is not complete: {status}")

    result = job.result(timeout=120, poll_interval=1.0)
    counts_list = job_result_to_counts(result)

    usage = None
    try:
        usage = float(job.usage(partial=False))
    except Exception:
        pass

    try:
        creation_date = str(job.creation_date)
    except Exception:
        creation_date = None

    payload = {
        "experiment": "sweet_spot_vs_random",
        "level": 2,
        "backend": backend,
        "job_id": job_id,
        "shots": SHOTS_EXPECTED,
        "qpu_usage_seconds": usage,
        "creation_date": creation_date,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "raw_counts_by_circuit_index": {
            str(i): counts_list[i] for i in range(len(counts_list))
        },
    }

    raw_path = OUT_DIR / f"{backend}_{job_id}_raw_retrieved.json"
    raw_path.write_text(json.dumps(payload, indent=2))
    print("Saved:", raw_path)
    return payload

# Uncomment and run only if one or more result JSONs are missing.
#
# from qiskit_ibm_runtime import QiskitRuntimeService
# service = QiskitRuntimeService()
#
# if not KNOWN_JOB_IDS["ibm_kingston"]:
#     KNOWN_JOB_IDS["ibm_kingston"] = discover_recent_kingston_jobs(service)
#
# for backend, ids in KNOWN_JOB_IDS.items():
#     for job_id in ids:
#         retrieve_job_to_raw_json(service, backend, job_id)


## 4. Load the final hardware results

This loader accepts either:

1. the final runner's per-backend result JSON;
2. a combined hardware result JSON;
3. the raw counts cache created by the IBM recovery cell.

It does not silently mix the earlier August Model-A/B/C experiment with the September Temporal-30-day experiment.


In [11]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_result_payload(payload, backend):
    rows = []

    if isinstance(payload, dict) and isinstance(payload.get("results"), list):
        for r in payload["results"]:
            rr = dict(r)
            rr["backend"] = rr.get("backend", backend)
            if "ghz_fidelity" not in rr and "fidelity" in rr:
                rr["ghz_fidelity"] = rr["fidelity"]
            rows.append(rr)
        return rows

    if isinstance(payload, dict) and "raw_counts_by_circuit_index" in payload:
        for k, counts in payload["raw_counts_by_circuit_index"].items():
            rows.append({
                "backend": backend,
                "circuit_index": int(k),
                "job_id": payload.get("job_id"),
                "counts": counts,
                "shots": payload.get("shots", SHOTS_EXPECTED),
                "ghz_fidelity": ghz_fidelity(counts),
            })
        return rows

    if isinstance(payload, list):
        for r in payload:
            rr = dict(r)
            rr["backend"] = rr.get("backend", backend)
            if "ghz_fidelity" not in rr and "fidelity" in rr:
                rr["ghz_fidelity"] = rr["fidelity"]
            rows.append(rr)
        return rows

    raise ValueError(f"Unrecognized result payload for {backend}")

hardware_rows = []

for backend, path in RESULT_FILES.items():
    if path is not None:
        payload = load_json(path)
        rows = normalize_result_payload(payload, backend)
        hardware_rows.extend(rows)

# Also pick up raw recovery files if no primary result file exists.
for backend in BACKENDS:
    if not any(r.get("backend") == backend for r in hardware_rows):
        raw_candidates = sorted(OUT_DIR.glob(f"{backend}_*_raw_retrieved.json"))
        for p in raw_candidates:
            hardware_rows.extend(normalize_result_payload(load_json(p), backend))

hardware = pd.DataFrame(hardware_rows)

if hardware.empty:
    print("No final hardware result files are currently available.")
    print("Run the IBM recovery cell above, then rerun this cell.")
else:
    print("Loaded rows:", len(hardware))
    display(hardware.head(20))


Loaded rows: 33


,backend,circuit_index,group,trial,model,physical_qubits,mean_viability,shots,job_id,counts,ghz_fidelity,transpiled_depth
0,ibm_fez,0,sweet_spot,0,temporal_30day,"[13, 14, 15]",0.921130,256,dafgfb5nj4cs73agbfo0,"{'011': 5, '000': 131, '111': 110, '100': 6, '...",0.941406,12
1,ibm_fez,1,random_control,1,random,"[63, 64, 65]",NaN,256,dafgfb5nj4cs73agbfo0,"{'111': 79, '100': 14, '000': 128, '011': 13, ...",0.808594,12
2,ibm_fez,2,random_control,2,random,"[82, 83, 96]",NaN,256,dafgfb5nj4cs73agbfo0,"{'000': 101, '111': 92, '010': 13, '100': 7, '...",0.753906,12
3,ibm_fez,3,random_control,3,random,"[109, 118, 129]",NaN,256,dafgfb5nj4cs73agbfo0,"{'111': 117, '000': 125, '100': 6, '011': 3, '...",0.945312,12
4,ibm_fez,4,random_control,4,random,"[104, 105, 106]",NaN,256,dafgfb5nj4cs73agbfo0,"{'111': 121, '000': 129, '011': 3, '101': 1, '...",0.976562,12
5,ibm_fez,5,random_control,5,random,"[100, 101, 102]",NaN,256,dafgfb5nj4cs73agbfo0,"{'000': 107, '111': 101, '100': 17, '011': 14,...",0.812500,12
6,ibm_fez,6,random_control,6,random,"[60, 61, 62]",NaN,256,dafgfb5nj4cs73agbfo0,"{'000': 113, '111': 109, '101': 4, '110': 8, '...",0.867188,12
7,ibm_fez,7,random_control,7,random,"[91, 92, 98]",NaN,256,dafgfb5nj4cs73agbfo0,"{'111': 114, '000': 127, '011': 2, '110': 3, '...",0.941406,18
8,ibm_fez,8,random_control,8,random,"[24, 25, 26]",NaN,256,dafgfb5nj4cs73agbfo0,"{'111': 104, '000': 131, '011': 1, '001': 7, '...",0.917969,12
9,ibm_fez,9,random_control,9,random,"[67, 68, 69]",NaN,256,dafgfb5nj4cs73agbfo0,"{'001': 4, '000': 122, '111': 114, '100': 5, '...",0.921875,12


## 5. Attach the final September candidate metadata

The hardware result order is:

1. Sweet-Spot
2. Random control 1
3. Random control 2
4. …
5. Random control 10

The notebook uses the final September candidate file if it is available. If it is not available, the known final candidate definitions are provided below as a reproducibility fallback.

The fallback is used only to label the already-executed circuits; it does not alter any measured hardware result.


In [12]:
# If the runner result already contains group/qubit/depth metadata, preserve it.
# Otherwise attach metadata according to circuit index.
def attach_metadata(df):
    out = df.copy()

    if "circuit_index" not in out.columns:
        out["circuit_index"] = out.groupby("job_id").cumcount()

    if "group" not in out.columns:
        out["group"] = np.nan
    if "physical_qubits" not in out.columns:
        out["physical_qubits"] = np.nan
    if "mean_viability" not in out.columns:
        out["mean_viability"] = np.nan

    for idx, row in out.iterrows():
        backend = row["backend"]
        ci = int(row["circuit_index"])
        inferred_group = "sweet_spot" if ci == 0 else f"random_{ci}"
        meta = FINAL_CANDIDATES.get(backend, {}).get(inferred_group, {})

        if pd.isna(row["group"]) or str(row["group"]).strip() in {"", "nan", "None"}:
            out.at[idx, "group"] = inferred_group

        if pd.isna(row["physical_qubits"]) or str(row["physical_qubits"]).strip() in {"", "nan", "None"}:
            out.at[idx, "physical_qubits"] = str(meta.get("qubits"))

        if pd.isna(row["mean_viability"]) and "mean_viability" in meta:
            out.at[idx, "mean_viability"] = meta["mean_viability"]

    out["group"] = out["group"].astype(str).replace({
        "sweet-spot": "sweet_spot",
        "sweet spot": "sweet_spot",
    })
    out["is_sweet_spot"] = out["group"].eq("sweet_spot") | out["group"].eq("model_selected")

    if "model" in out.columns:
        out.loc[
            out["model"].astype(str).str.contains("temporal_30day|sweet", case=False, na=False),
            "is_sweet_spot"
        ] = True

    out["ghz_fidelity"] = out["ghz_fidelity"].astype(float)
    return out

hardware = attach_metadata(hardware)

dedupe_cols = [c for c in ["backend", "job_id", "circuit_index"] if c in hardware.columns]
if dedupe_cols:
    hardware = hardware.drop_duplicates(dedupe_cols).reset_index(drop=True)

display(hardware.sort_values(["backend", "job_id", "circuit_index"]).head(40))


NameError: name 'FINAL_CANDIDATES' is not defined

## 6. Primary final-experiment dataset

The primary comparison uses:

- one Sweet-Spot circuit per backend;
- ten random controls per backend;
- 256 shots per circuit.

If two Kingston jobs are present, the newest completed Kingston job is marked as the **primary Kingston run**, while the other is treated as a replication run.



In [ ]:
# Determine primary job per backend.
# For Kingston, the newest completed job is the primary final run and the older
# completed job is treated as a same-backend replication. The selection uses
# creation timestamps when available, not lexicographic job IDs.

primary_job = {}
replication_jobs = {}

# If raw IBM-retrieval caches contain creation timestamps, use them.
job_timestamp = {}
for p in OUT_DIR.glob("*_raw_retrieved.json"):
    try:
        payload = load_json(p)
        jid = payload.get("job_id")
        created = payload.get("creation_date")
        if jid and created:
            job_timestamp[str(jid)] = pd.to_datetime(created, errors="coerce")
    except Exception:
        pass

for backend in BACKENDS:
    sub = hardware[hardware["backend"] == backend].copy()
    if sub.empty:
        continue

    ids = list(sub["job_id"].dropna().astype(str).unique())

    if len(ids) == 1:
        primary_job[backend] = ids[0]
        replication_jobs[backend] = []
        continue

    # Known fixed ordering for the two Kingston runs can be supplied explicitly.
    # Otherwise use cached IBM creation timestamps; newest = primary.
    if backend == "ibm_kingston":
        ranked = sorted(
            ids,
            key=lambda jid: (
                job_timestamp.get(jid, pd.Timestamp.min)
                if pd.notna(job_timestamp.get(jid, pd.NaT))
                else pd.Timestamp.min
            )
        )
        if len(set(job_timestamp).intersection(ids)) >= 2:
            primary_job[backend] = ranked[-1]
            replication_jobs[backend] = ranked[:-1]
        else:
            # If timestamps are unavailable, require the user to set the
            # primary job explicitly rather than silently choosing by ID.
            print("Kingston has multiple jobs but no creation timestamps were loaded.")
            print("Set PRIMARY_KINGSTON_JOB_ID manually in this cell if needed.")
            PRIMARY_KINGSTON_JOB_ID = None
            if PRIMARY_KINGSTON_JOB_ID:
                primary_job[backend] = PRIMARY_KINGSTON_JOB_ID
                replication_jobs[backend] = [x for x in ids if x != PRIMARY_KINGSTON_JOB_ID]
            else:
                # Deterministic fallback: the first job in the loaded result
                # order is used, and the notebook prints the ambiguity.
                primary_job[backend] = ids[-1]
                replication_jobs[backend] = ids[:-1]
                print("WARNING: Kingston primary selection is ambiguous; rerun IBM recovery to attach timestamps.")
    else:
        primary_job[backend] = ids[-1]
        replication_jobs[backend] = ids[:-1]

print("Primary jobs:")
for b, j in primary_job.items():
    print(" ", b, "->", j)

print("\nReplication jobs:")
for b, js in replication_jobs.items():
    print(" ", b, "->", js)


In [ ]:
primary = pd.concat(
    [hardware[(hardware["backend"] == b) & (hardware["job_id"].astype(str) == str(j))]
     for b, j in primary_job.items()],
    ignore_index=True
)

# A final job should contain exactly 11 circuits.
qc = primary.groupby("backend").agg(
    circuits=("circuit_index", "nunique"),
    shots=("shots", "first") if "shots" in primary.columns else ("circuit_index", "size"),
    min_fidelity=("ghz_fidelity", "min"),
    max_fidelity=("ghz_fidelity", "max"),
)
display(qc)

assert set(primary["backend"]) == set([b for b in BACKENDS if b in primary["backend"].unique()])
assert all(primary.groupby("backend")["circuit_index"].nunique() == EXPECTED_CIRCUITS_PER_JOB),     "A primary backend job does not contain exactly 11 circuits."

print("Primary final experiment dataset validated.")


# 7. Primary hardware outcome: Sweet-Spot vs random controls

This is the central test.

For each backend we calculate:

- Sweet-Spot GHZ fidelity
- random-control mean / median / standard deviation
- absolute advantage: `F_sweet - mean(F_random)`
- relative improvement
- Sweet-Spot rank among 11 circuits
- percentile among all 11 circuits
- fraction of random controls beaten

The correct statistical unit for the main claim is the **backend**, not the 256 individual shots. Shots provide the measurement uncertainty for each fidelity; they do not create 256 independent backend-level experiments.


In [ ]:
def wilson_ci(k, n, z=1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    phat = k / n
    denom = 1 + z*z/n
    center = (phat + z*z/(2*n)) / denom
    half = z * math.sqrt(phat*(1-phat)/n + z*z/(4*n*n)) / denom
    return max(0, center-half), min(1, center+half)

def summarize_backend(df):
    rows = []
    for backend, g in df.groupby("backend"):
        sweet = g[g["is_sweet_spot"]].iloc[0]
        controls = g[~g["is_sweet_spot"]].copy()

        s = float(sweet["ghz_fidelity"])
        cmean = float(controls["ghz_fidelity"].mean())
        cmedian = float(controls["ghz_fidelity"].median())
        cstd = float(controls["ghz_fidelity"].std(ddof=1))
        diff = s - cmean
        rel = diff / cmean if cmean else np.nan

        all_rank = int((-g["ghz_fidelity"]).rank(method="min").loc[sweet.name])
        beaten = int((controls["ghz_fidelity"] < s).sum())
        ties = int((controls["ghz_fidelity"] == s).sum())

        shots = int(sweet.get("shots", SHOTS_EXPECTED))
        k = int(round(s * shots))
        lo, hi = wilson_ci(k, shots)

        rows.append({
            "backend": backend,
            "sweet_fidelity": s,
            "sweet_ci_low": lo,
            "sweet_ci_high": hi,
            "random_mean": cmean,
            "random_median": cmedian,
            "random_std": cstd,
            "absolute_advantage": diff,
            "relative_improvement": rel,
            "sweet_rank_1_best": all_rank,
            "sweet_percentile_among_11": 100 * (1 - (all_rank - 1) / max(len(g)-1, 1)),
            "controls_beaten": beaten,
            "control_ties": ties,
            "n_controls": len(controls),
        })
    return pd.DataFrame(rows)

backend_summary = summarize_backend(primary)
display(backend_summary.style.format({
    "sweet_fidelity": "{:.4f}",
    "sweet_ci_low": "{:.4f}",
    "sweet_ci_high": "{:.4f}",
    "random_mean": "{:.4f}",
    "random_median": "{:.4f}",
    "random_std": "{:.4f}",
    "absolute_advantage": "{:.4f}",
    "relative_improvement": "{:.2%}",
    "sweet_percentile_among_11": "{:.1f}",
}))


## 8. Circuit-level ranking

A useful sanity check is whether the selected Sweet-Spot circuit actually ranks near the top of the measured 11 circuits.

This is more informative than only comparing the mean random baseline.


In [ ]:
ranking = primary.copy()
ranking["rank_1_best"] = ranking.groupby("backend")["ghz_fidelity"].rank(method="min", ascending=False).astype(int)
ranking["is_control"] = ~ranking["is_sweet_spot"]
ranking = ranking.sort_values(["backend", "rank_1_best"])

display_cols = [c for c in [
    "backend", "rank_1_best", "group", "physical_qubits",
    "ghz_fidelity", "mean_viability", "transpiled_depth", "job_id"
] if c in ranking.columns]
display(ranking[display_cols])


# 9. Paired backend-level evidence

The main experimental design is blocked by backend: every backend has one selected chain and ten random controls.

We therefore use a **paired backend comparison**:

`Sweet-Spot fidelity - mean random-control fidelity`

across FEZ, Kingston, and Marrakesh.

With only three backends, inferential p-values are necessarily low-powered. The notebook reports them, but the effect size and consistency across backends should carry more weight than a conventional significance threshold.


In [ ]:
paired = backend_summary[["backend", "sweet_fidelity", "random_mean", "absolute_advantage"]].copy()

print("Backend-level advantages:")
display(paired)

diffs = paired["absolute_advantage"].to_numpy(dtype=float)

if len(diffs) >= 2:
    print("One-sample t-test on backend advantages:")
    print(stats.ttest_1samp(diffs, 0.0))

    print("\nWilcoxon signed-rank test on backend advantages:")
    try:
        print(stats.wilcoxon(diffs, alternative="greater", zero_method="wilcox"))
    except Exception as e:
        print("Wilcoxon unavailable:", e)

print("\nMean advantage:", np.mean(diffs))
print("Median advantage:", np.median(diffs))
print("All backends positive:", bool(np.all(diffs > 0)))


# 10. Random-control permutation test

Because there are only three backend-level observations, we also use a small exact-style sign/permutation calculation.

For each backend, the random controls are treated as the empirical null distribution. We ask:

> How often would a randomly selected control chain have a fidelity at least as high as the Sweet-Spot chain?

This gives an intuitive empirical p-value / exceedance rate without pretending that the 30 random controls are fully independent of backend.


In [ ]:
rows = []
for backend, g in primary.groupby("backend"):
    sweet = float(g.loc[g["is_sweet_spot"], "ghz_fidelity"].iloc[0])
    controls = g.loc[~g["is_sweet_spot"], "ghz_fidelity"].to_numpy(float)

    exceed = int(np.sum(controls >= sweet))
    empirical_p = (exceed + 1) / (len(controls) + 1)

    rows.append({
        "backend": backend,
        "sweet_fidelity": sweet,
        "n_controls": len(controls),
        "controls_ge_sweet": exceed,
        "empirical_one_sided_p": empirical_p,
        "sweet_beats_all_controls": exceed == 0,
    })

empirical = pd.DataFrame(rows)
display(empirical)


# 11. Bootstrap confidence intervals

Two bootstrap quantities are useful:

1. the **within-backend random-control mean** uncertainty;
2. the **backend-level Sweet-Spot advantage** uncertainty.

The first is descriptive. The second is the more conservative experiment-level view.

The bootstrap does not manufacture independent backend observations; it is used to characterize the observed finite sample.


In [ ]:
rng = np.random.default_rng(20260909)

bootstrap_rows = []

for backend, g in primary.groupby("backend"):
    sweet = float(g.loc[g["is_sweet_spot"], "ghz_fidelity"].iloc[0])
    controls = g.loc[~g["is_sweet_spot"], "ghz_fidelity"].to_numpy(float)

    boot = np.empty(20000)
    for i in range(len(boot)):
        sample = rng.choice(controls, size=len(controls), replace=True)
        boot[i] = sweet - np.mean(sample)

    bootstrap_rows.append({
        "backend": backend,
        "observed_advantage": sweet - np.mean(controls),
        "bootstrap_ci_low": np.quantile(boot, 0.025),
        "bootstrap_ci_high": np.quantile(boot, 0.975),
        "bootstrap_prob_advantage_gt_0": np.mean(boot > 0),
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df)


# 12. Shot-level measurement uncertainty

Each circuit used 256 shots. The GHZ fidelity is a binomial proportion because the event is:

`000 or 111`

and the complement is all other bitstrings.

We report Wilson intervals for each circuit. These intervals describe **shot noise**, not model uncertainty or backend-to-backend variation.


In [ ]:
shot_rows = []
for _, r in primary.iterrows():
    counts = r.get("counts", {})
    if isinstance(counts, str):
        try:
            counts = json.loads(counts.replace("'", '"'))
        except Exception:
            counts = {}

    shots = int(r.get("shots", SHOTS_EXPECTED))
    k = int(counts.get("000", 0)) + int(counts.get("111", 0))
    lo, hi = wilson_ci(k, shots)

    shot_rows.append({
        "backend": r["backend"],
        "group": r["group"],
        "fidelity": r["ghz_fidelity"],
        "shots": shots,
        "ghz_success_counts": k,
        "ci_low": lo,
        "ci_high": hi,
    })

shot_uncertainty = pd.DataFrame(shot_rows)
display(shot_uncertainty)


# 13. Depth confound / routing sensitivity

This is an important scientific caveat.

The final selector chose physical topology-valid chains, but some random controls required additional transpiler routing. The recorded experiment contains examples where:

- Sweet-Spot depth = 12
- some random controls = 17 or 18

A lower fidelity on a deeper random circuit could partly reflect **extra gates / routing**, rather than poorer physical qubits.

Therefore the strongest conclusion is not simply:

> "The model beats random."

It is:

> "The model-selected Sweet-Spot chains produced better observed GHZ fidelity than the sampled topology-valid random controls, but the result must be interpreted with the transpiled-depth imbalance as a potential confound."

This notebook therefore reports a depth-matched sensitivity analysis whenever enough controls have the same depth as the Sweet-Spot.


In [ ]:
if "transpiled_depth" in primary.columns:
    depth_summary = (
        primary.groupby(["backend", "is_sweet_spot"])["transpiled_depth"]
        .agg(["count", "min", "median", "max", "mean"])
        .reset_index()
    )
    display(depth_summary)

    sensitivity_rows = []
    for backend, g in primary.groupby("backend"):
        sweet = g[g["is_sweet_spot"]].iloc[0]
        controls = g[~g["is_sweet_spot"]].copy()
        same_depth = controls[controls["transpiled_depth"] == sweet["transpiled_depth"]]

        if len(same_depth):
            same_mean = same_depth["ghz_fidelity"].mean()
            sensitivity_rows.append({
                "backend": backend,
                "sweet_depth": int(sweet["transpiled_depth"]),
                "same_depth_controls": len(same_depth),
                "sweet_fidelity": float(sweet["ghz_fidelity"]),
                "same_depth_random_mean": float(same_mean),
                "same_depth_advantage": float(sweet["ghz_fidelity"] - same_mean),
            })
        else:
            sensitivity_rows.append({
                "backend": backend,
                "sweet_depth": int(sweet["transpiled_depth"]),
                "same_depth_controls": 0,
                "sweet_fidelity": float(sweet["ghz_fidelity"]),
                "same_depth_random_mean": np.nan,
                "same_depth_advantage": np.nan,
            })

    depth_sensitivity = pd.DataFrame(sensitivity_rows)
    display(depth_sensitivity)
else:
    print("No transpiled-depth metadata available in the result files.")


# 14. Kingston repeatability

Because Kingston was executed twice, this is valuable additional evidence.

We compare the two completed Kingston jobs circuit-by-circuit using the same 11 circuit labels.

This does **not** count as two independent backend replicates. Instead, it asks whether the selected-versus-control pattern is reasonably reproducible on the same backend.


In [ ]:
kingston = hardware[hardware["backend"] == "ibm_kingston"].copy()

kingston_jobs = list(kingston["job_id"].dropna().astype(str).unique())

if len(kingston_jobs) >= 2:
    rep_tables = []
    for jid in kingston_jobs:
        tmp = kingston[kingston["job_id"].astype(str) == jid].copy()
        tmp = tmp.sort_values("circuit_index")
        tmp["circuit_label"] = ["sweet_spot" if i == 0 else f"random_{i}" for i in range(len(tmp))]
        rep_tables.append(tmp[["circuit_index", "circuit_label", "ghz_fidelity"]].rename(
            columns={"ghz_fidelity": f"fidelity_{jid}"}
        ).set_index("circuit_index"))

    rep = rep_tables[0].join(rep_tables[1], how="inner")
    cols = [c for c in rep.columns if c.startswith("fidelity_")]
    if len(cols) == 2:
        rep["difference"] = rep[cols[1]] - rep[cols[0]]
        rep["abs_difference"] = rep["difference"].abs()
        print("Kingston repeatability:")
        display(rep)

        print("Pearson correlation:", rep[cols].corr().iloc[0,1])
        print("Mean absolute circuit-level difference:", rep["abs_difference"].mean())
else:
    print("Fewer than two Kingston jobs are loaded; repeatability analysis is skipped.")


# 15. Temporal sweet-spot model comparison

The final model-selection question is separate from the hardware experiment.

We compare:

- 1-day
- 3-day
- 7-day
- 30-day

using the prospective temporal model-selection results.

The selection criteria were:

1. cross-backend LOBO AUC;
2. minimum backend AUC;
3. ranking stability across prediction dates.

A model with a slightly higher instantaneous viable rate is **not automatically better**. The goal is stable, generalizable ranking of physical qubits/chains.


In [ ]:
MODEL_META_PATHS = [
    PROJECT_ROOT / "temporal_sweet_spot_final_sep6_models.csv",
    Path("/mnt/data/temporal_sweet_spot_final_sep6_models.csv"),
]
MODEL_META_PATH = next((p for p in MODEL_META_PATHS if p.exists()), None)

if MODEL_META_PATH:
    model_meta = pd.read_csv(MODEL_META_PATH)
    cols = [
        "model_name", "window_days", "training_rows",
        "training_viable_rate", "lobo_mean_auc", "lobo_min_auc"
    ]
    display(model_meta[cols].sort_values("window_days"))
else:
    model_meta = pd.DataFrame([
        {"model_name":"temporal_1day","window_days":1,"training_rows":466,"training_viable_rate":0.624464,"lobo_mean_auc":0.891779,"lobo_min_auc":0.870588},
        {"model_name":"temporal_3day","window_days":3,"training_rows":1398,"training_viable_rate":0.623033,"lobo_mean_auc":0.876596,"lobo_min_auc":0.864435},
        {"model_name":"temporal_7day","window_days":7,"training_rows":3262,"training_viable_rate":0.624770,"lobo_mean_auc":0.870131,"lobo_min_auc":0.853088},
        {"model_name":"temporal_30day","window_days":30,"training_rows":13980,"training_viable_rate":0.605722,"lobo_mean_auc":0.893915,"lobo_min_auc":0.864922},
    ])
    print("Using the recorded final temporal-model metrics.")
    display(model_meta)


### Why the 30-day model won

The 30-day model did **not** win because it had the highest training viable rate.

In fact, its training viable rate was lower than the 1/3/7-day windows.

It won because it gave the strongest combination of:

- highest mean LOBO AUC among the tested windows;
- strong worst-backend AUC;
- substantially better ranking stability across prediction dates.

The 7-day model is therefore a useful comparison, but its mean LOBO AUC and stability were weaker.

This distinction is central to the definition of a temporal "sweet spot": **the useful window is the one that gives the most reliable prospective ranking, not merely the largest fraction of positive training labels.**


In [ ]:
RANKING_STABILITY_PATHS = [
    PROJECT_ROOT / "temporal_sweet_spot_ranking_stability.csv",
    Path("/mnt/data/temporal_sweet_spot_ranking_stability.csv"),
]
RANKING_STABILITY_PATH = next((p for p in RANKING_STABILITY_PATHS if p.exists()), None)

if RANKING_STABILITY_PATH:
    stability = pd.read_csv(RANKING_STABILITY_PATH)
    stability_summary = (
        stability.groupby("model_name")
        .agg(
            avg_top10_intersection=("top10_intersection", "mean"),
            avg_top10_jaccard=("top10_jaccard", "mean"),
        )
        .reset_index()
    )
else:
    stability_summary = pd.DataFrame([
        {"model_name":"temporal_1day","avg_top10_intersection":3.8889,"avg_top10_jaccard":0.246421},
        {"model_name":"temporal_3day","avg_top10_intersection":5.7778,"avg_top10_jaccard":0.422640},
        {"model_name":"temporal_7day","avg_top10_intersection":5.7778,"avg_top10_jaccard":0.440862},
        {"model_name":"temporal_30day","avg_top10_intersection":7.5556,"avg_top10_jaccard":0.625967},
    ])

display(stability_summary.sort_values("model_name"))

model_compare = model_meta.merge(stability_summary, on="model_name", how="left")
model_compare["rank_by_mean_auc"] = model_compare["lobo_mean_auc"].rank(ascending=False, method="min").astype(int)
model_compare["rank_by_stability"] = model_compare["avg_top10_jaccard"].rank(ascending=False, method="min").astype(int)

display(model_compare.sort_values("window_days"))


In [ ]:
# Explicit 7-day vs 30-day comparison.
m7 = model_compare[model_compare["window_days"] == 7].iloc[0]
m30 = model_compare[model_compare["window_days"] == 30].iloc[0]

comparison_7_vs_30 = pd.DataFrame([{
    "metric": "Mean LOBO AUC",
    "7_day": m7["lobo_mean_auc"],
    "30_day": m30["lobo_mean_auc"],
    "30_day_minus_7_day": m30["lobo_mean_auc"] - m7["lobo_mean_auc"],
}, {
    "metric": "Minimum LOBO AUC",
    "7_day": m7["lobo_min_auc"],
    "30_day": m30["lobo_min_auc"],
    "30_day_minus_7_day": m30["lobo_min_auc"] - m7["lobo_min_auc"],
}, {
    "metric": "Mean Top-10 Jaccard",
    "7_day": m7["avg_top10_jaccard"],
    "30_day": m30["avg_top10_jaccard"],
    "30_day_minus_7_day": m30["avg_top10_jaccard"] - m7["avg_top10_jaccard"],
}, {
    "metric": "Training viable rate",
    "7_day": m7["training_viable_rate"],
    "30_day": m30["training_viable_rate"],
    "30_day_minus_7_day": m30["training_viable_rate"] - m7["training_viable_rate"],
}])

display(comparison_7_vs_30)


# 16. Does the model actually select better qubits?

There are two distinct claims we can test.

### Claim A — predictive validity

The temporal model predicts viability better than chance across held-out backends.

Evidence: LOBO AUC.

### Claim B — experimental utility

The model-selected physical chain produces higher downstream GHZ fidelity than topology-valid random chains.

Evidence: the September hardware comparison.

These claims should not be conflated. A model can have good AUC without producing a large hardware effect, and a hardware effect can be observed even when a model is imperfect.


In [ ]:
# A compact decision table for the final claim.
decision_rows = []

for _, r in backend_summary.iterrows():
    decision_rows.append({
        "backend": r["backend"],
        "sweet_spot_fidelity": r["sweet_fidelity"],
        "random_control_mean": r["random_mean"],
        "advantage": r["absolute_advantage"],
        "relative_improvement": r["relative_improvement"],
        "sweet_spot_rank": int(r["sweet_rank_1_best"]),
        "controls_beaten": f"{int(r['controls_beaten'])}/{int(r['n_controls'])}",
        "beats_all_controls": int(r["controls_beaten"]) == int(r["n_controls"]),
    })

decision = pd.DataFrame(decision_rows)
display(decision)

print("\nOVERALL HARDWARE DECISION")
if not decision.empty:
    positive_fraction = (decision["advantage"] > 0).mean()
    print(f"Backends where Sweet-Spot > random mean: {int((decision['advantage'] > 0).sum())}/{len(decision)}")
    print(f"Fraction of backends positive: {positive_fraction:.1%}")
    print(f"Mean backend advantage: {decision['advantage'].mean():.4f}")
    print(f"Median backend advantage: {decision['advantage'].median():.4f}")
    print(f"Sweet-Spot beats every sampled control on all backends: {bool(decision['beats_all_controls'].all())}")


# 17. Model-prediction vs hardware-fidelity relationship

For the selected 30-day model, the predicted chain viability can be compared with observed GHZ fidelity.

This is exploratory because the sample contains only three Sweet-Spot chains.

We should **not** expect a perfect numerical correspondence: the model predicts telemetry-based viability, while the circuit experiment measures a particular GHZ workload under a particular transpilation and calibration state.


In [ ]:
pred_vs_hw = backend_summary.merge(
    pd.DataFrame([
        {"backend": "ibm_fez", "mean_viability": 0.921130},
        {"backend": "ibm_kingston", "mean_viability": 0.960623},
        {"backend": "ibm_marrakesh", "mean_viability": 0.927097},
    ]),
    on="backend",
    how="left",
)

display(pred_vs_hw)

if len(pred_vs_hw) >= 3:
    print("Spearman correlation:", stats.spearmanr(pred_vs_hw["mean_viability"], pred_vs_hw["sweet_fidelity"]))
    print("Pearson correlation:", stats.pearsonr(pred_vs_hw["mean_viability"], pred_vs_hw["sweet_fidelity"]))


# 18. Required plots

The notebook generates:

1. Sweet-Spot vs random mean fidelity by backend.
2. All 11 circuit fidelities by backend.
3. Sweet-Spot advantage over random controls.
4. Temporal-window mean LOBO AUC.
5. Temporal-window ranking stability.
6. 7-day vs 30-day comparison.
7. Model viability vs hardware fidelity.
8. Kingston repeatability when both jobs are available.


In [ ]:
# Plot 1: Sweet-Spot vs random mean
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(backend_summary))
w = 0.36
ax.bar(x - w/2, backend_summary["sweet_fidelity"], width=w, label="Sweet-Spot")
ax.bar(x + w/2, backend_summary["random_mean"], width=w, label="Random mean")
ax.set_xticks(x)
ax.set_xticklabels(backend_summary["backend"].str.replace("ibm_", "", regex=False))
ax.set_ylabel("GHZ fidelity")
ax.set_title("Final hardware: Sweet-Spot vs random controls")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "01_sweet_vs_random_mean.png", dpi=220)
plt.show()

# Plot 2: all circuits
fig, ax = plt.subplots(figsize=(12, 6))
for backend, g in primary.groupby("backend"):
    g = g.sort_values("circuit_index")
    labels = ["Sweet-Spot"] + [f"R{i}" for i in range(1, len(g))]
    xx = np.arange(len(g))
    ax.plot(xx, g["ghz_fidelity"], marker="o", label=backend.replace("ibm_", ""))
ax.set_xticks(range(EXPECTED_CIRCUITS_PER_JOB))
ax.set_xticklabels(["SS"] + [f"R{i}" for i in range(1, 11)])
ax.set_ylabel("GHZ fidelity")
ax.set_title("All final circuits by backend")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "02_all_circuit_fidelities.png", dpi=220)
plt.show()

# Plot 3: advantage
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(backend_summary["backend"].str.replace("ibm_", "", regex=False), backend_summary["absolute_advantage"])
ax.axhline(0, linewidth=1)
ax.set_ylabel("Sweet-Spot fidelity − random mean")
ax.set_title("Sweet-Spot advantage")
fig.tight_layout()
fig.savefig(OUT_DIR / "03_sweet_spot_advantage.png", dpi=220)
plt.show()


In [ ]:
# Plot 4: temporal model AUC
fig, ax = plt.subplots(figsize=(8, 5))
tmp = model_compare.sort_values("window_days")
ax.plot(tmp["window_days"], tmp["lobo_mean_auc"], marker="o", label="Mean LOBO AUC")
ax.plot(tmp["window_days"], tmp["lobo_min_auc"], marker="o", label="Minimum LOBO AUC")
ax.set_xlabel("Training window (days)")
ax.set_ylabel("AUC")
ax.set_title("Temporal window: predictive performance")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "04_temporal_auc.png", dpi=220)
plt.show()

# Plot 5: stability
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tmp["window_days"], tmp["avg_top10_jaccard"], marker="o")
ax.set_xlabel("Training window (days)")
ax.set_ylabel("Mean Top-10 Jaccard")
ax.set_title("Temporal window: ranking stability")
fig.tight_layout()
fig.savefig(OUT_DIR / "05_temporal_stability.png", dpi=220)
plt.show()

# Plot 6: 7d vs 30d metric comparison
fig, ax = plt.subplots(figsize=(9, 5))
metrics = ["Mean LOBO AUC", "Minimum LOBO AUC", "Mean Top-10 Jaccard"]
v7 = [m7["lobo_mean_auc"], m7["lobo_min_auc"], m7["avg_top10_jaccard"]]
v30 = [m30["lobo_mean_auc"], m30["lobo_min_auc"], m30["avg_top10_jaccard"]]
xx = np.arange(len(metrics))
ax.bar(xx - 0.18, v7, width=0.36, label="7-day")
ax.bar(xx + 0.18, v30, width=0.36, label="30-day")
ax.set_xticks(xx)
ax.set_xticklabels(metrics, rotation=15)
ax.set_ylabel("Score")
ax.set_title("Why 30-day was selected over 7-day")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "06_7day_vs_30day.png", dpi=220)
plt.show()

# Plot 7: viability vs observed Sweet-Spot fidelity
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(pred_vs_hw["mean_viability"], pred_vs_hw["sweet_fidelity"], s=80)
for _, r in pred_vs_hw.iterrows():
    ax.annotate(r["backend"].replace("ibm_", ""), (r["mean_viability"], r["sweet_fidelity"]), xytext=(5,5), textcoords="offset points")
ax.set_xlabel("Predicted chain viability")
ax.set_ylabel("Observed GHZ fidelity")
ax.set_title("Predicted viability vs hardware fidelity")
fig.tight_layout()
fig.savefig(OUT_DIR / "07_viability_vs_hardware_fidelity.png", dpi=220)
plt.show()


# 19. Automated final interpretation

This cell produces a concise research conclusion from the measured values instead of hard-coding a desired outcome.

It distinguishes:

- **supported**: Sweet-Spot consistently beats the sampled random mean;
- **partially supported**: positive effect is present but not consistent across backends;
- **not supported**: no positive aggregate effect;
- **qualified**: depth imbalance prevents a clean causal claim.


In [ ]:
def final_interpretation(decision, depth_sensitivity=None):
    if decision.empty:
        return "No hardware data loaded."

    n = len(decision)
    positive = int((decision["advantage"] > 0).sum())
    mean_adv = float(decision["advantage"].mean())

    if positive == n and mean_adv > 0:
        verdict = "SUPPORTED WITH EXPERIMENTAL QUALIFIERS"
        sentence = (
            f"The Temporal-30-day Sweet-Spot chain outperformed the sampled random-control mean "
            f"on all {n} tested backends, with a mean backend-level advantage of {mean_adv:.4f} fidelity."
        )
    elif positive > n/2 and mean_adv > 0:
        verdict = "PARTIALLY SUPPORTED"
        sentence = (
            f"The Sweet-Spot produced a positive mean advantage ({mean_adv:.4f}), "
            f"but the effect was not positive on every backend."
        )
    else:
        verdict = "NOT SUPPORTED BY THIS HARDWARE SAMPLE"
        sentence = (
            f"The measured hardware sample does not provide consistent evidence that the "
            f"Sweet-Spot beats random controls; mean advantage = {mean_adv:.4f}."
        )

    depth_note = ""
    if depth_sensitivity is not None and not depth_sensitivity.empty:
        if "same_depth_controls" in depth_sensitivity:
            available = int((depth_sensitivity["same_depth_controls"] > 0).sum())
            if available < n:
                depth_note = (
                    " A clean causal interpretation is limited because not every backend has "
                    "enough same-depth random controls; transpilation/routing depth is therefore a confound."
                )

    return f"### {verdict}\n\n{sentence}{depth_note}\n\n"            "The result demonstrates downstream workload performance for the tested GHZ circuit; "            "it does not prove universal superiority across all circuits or all future calibrations."

print(final_interpretation(decision, globals().get("depth_sensitivity")))


# 20. Export all evaluation artifacts

The notebook saves:

- normalized hardware measurements;
- backend summary;
- circuit ranking;
- shot uncertainty;
- empirical control comparison;
- bootstrap analysis;
- depth sensitivity;
- temporal model comparison;
- 7-day vs 30-day comparison;
- plots.

These files can be used directly in the final project report.


In [ ]:
exports = {
    "hardware_primary.csv": primary,
    "hardware_all_loaded.csv": hardware,
    "backend_summary.csv": backend_summary,
    "circuit_ranking.csv": ranking,
    "shot_uncertainty.csv": shot_uncertainty,
    "empirical_control_test.csv": empirical,
    "bootstrap_advantage.csv": bootstrap_df,
    "temporal_model_comparison.csv": model_compare,
    "temporal_7day_vs_30day.csv": comparison_7_vs_30,
    "predicted_vs_hardware.csv": pred_vs_hw,
}

if "depth_sensitivity" in globals():
    exports["depth_sensitivity.csv"] = depth_sensitivity

for name, df in exports.items():
    if isinstance(df, pd.DataFrame):
        df.to_csv(OUT_DIR / name, index=False)

summary_json = {
    "experiment": "Temporal 30-day Sweet-Spot vs topology-valid random controls",
    "prediction_date": PREDICTION_DATE,
    "shots_per_circuit": SHOTS_EXPECTED,
    "circuits_per_backend": EXPECTED_CIRCUITS_PER_JOB,
    "primary_jobs": primary_job,
    "backend_summary": backend_summary.to_dict(orient="records"),
    "temporal_model_comparison": model_compare.to_dict(orient="records"),
    "seven_vs_thirty_day": comparison_7_vs_30.to_dict(orient="records"),
}

(OUT_DIR / "final_evaluation_summary.json").write_text(json.dumps(summary_json, indent=2, default=str))

print("Saved evaluation artifacts to:", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(" ", p.name)


# Final research conclusion

Use the generated tables and plots to answer the following questions explicitly in the report:

### What is the Sweet-Spot?

The Sweet-Spot is the topology-valid 3-qubit chain with the highest mean predicted viability under the selected Temporal-30-day model on the prediction date.

### Why 30 days instead of 7 days?

The 30-day window was selected because its prospective LOBO predictive performance and ranking stability were stronger than the 7-day window. The 7-day model had a slightly higher training viable rate, but training viable rate is not the selection objective.

### Did the model beat random?

The strongest supported statement is based on the actual hardware comparison:

> The model-selected Sweet-Spot chain is considered experimentally successful only to the extent that its measured GHZ fidelity exceeds the random-control distribution within the same backend.

The notebook reports this separately for FEZ, Kingston, and Marrakesh and then gives a backend-level aggregate.

### What prevents overclaiming?

- only three independent backends;
- 256 shots per circuit;
- finite random-control sample;
- possible transpilation-depth imbalance;
- one workload (GHZ fidelity), not all workloads;
- model viability and hardware fidelity are related but not identical quantities.

The final result should therefore be described as **evidence for the utility of telemetry-based Sweet-Spot selection**, not proof of universal optimality.
